# 🧪 Sesión 23 · Operaciones CRUD Transaccionales y Async Sessions

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué esperar a la base de datos no debe bloquear el servidor?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/libros  {"titulo":"Rayuela","anio":1963}  -> 201
GET  /api/libros                                  -> 200
GET  /api/libros/1                                -> 200
PUT  /api/libros/1  {"titulo":"Rayuela (edición)"} -> 200
DELETE /api/libros/1                              -> 204
GET  /api/libros/1                                -> 404
```

_La base de datos sigue siendo SQLite; lo que cambia es cómo la aplicación espera sus respuestas._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar un CRUD con SQLAlchemy asíncrono.
- Usar AsyncSession y aiosqlite.
- Verificar 201, 200, 404 y 204 en el mismo flujo.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Crear un libro → HTTP 201
- [ ] Listar los libros → HTTP 200
- [ ] Actualizar el libro → HTTP 200
- [ ] Eliminar el libro → HTTP 204
- [ ] El libro eliminado ya no existe → HTTP 404
- [ ] La persistencia es asíncrona

## 🧩 El reto

Implementa un CRUD completo con SQLAlchemy asíncrono y aiosqlite. El servidor no debe bloquearse mientras espera la base.

**Requisitos del contrato**

- Usa create_async_engine con sqlite+aiosqlite.
- Cada endpoint usa AsyncSession y await.
- POST responde 201; GET 200; PUT 200; DELETE 204.
- Un id inexistente responde 404.
- La tabla se crea al iniciar la aplicación.

> **Pista conceptual:** Si tu endpoint es `def` y usas una sesión síncrona, estás bloqueando el event loop. La regla es: motor async, sesión async, endpoint async.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion23.md` con la diferencia entre def y async def.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [4]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [7]:

%%writefile app.py

from fastapi import FastAPI, HTTPException, Depends, Response
from pydantic import BaseModel, Field, ConfigDict
from sqlalchemy import Integer, String, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy.ext.asyncio import (
    create_async_engine,
    AsyncSession,
    async_sessionmaker
)

# 1. Configuración de SQLite asíncrono
DATABASE_URL = "sqlite+aiosqlite:///./biblioteca.db"

engine = create_async_engine(
    DATABASE_URL,
    echo=False
)

SessionLocal = async_sessionmaker(
    engine,
    class_=AsyncSession,
    expire_on_commit=False
)

# 2. Base de datos
class Base(DeclarativeBase):
    pass


# 3. Tabla de libros
class Libro(Base):
    __tablename__ = "libros"

    id: Mapped[int] = mapped_column(
        Integer,
        primary_key=True,
        autoincrement=True
    )

    titulo: Mapped[str] = mapped_column(
        String(200),
        nullable=False
    )

    anio: Mapped[int] = mapped_column(
        Integer,
        nullable=False
    )


# 4. Modelos de entrada y salida
class LibroEntrada(BaseModel):
    titulo: str = Field(min_length=1, max_length=200)
    anio: int = Field(ge=1500, le=2026)


class LibroRespuesta(BaseModel):
    id: int
    titulo: str
    anio: int

    model_config = ConfigDict(from_attributes=True)


class ListaLibros(BaseModel):
    libros: list[LibroRespuesta]


# 5. Aplicación FastAPI
app = FastAPI(
    title="Biblioteca CRUD Asíncrona",
    version="1.0.0"
)


# 6. Obtener sesión y asegurar que exista la tabla
async def obtener_sesion():
    async with engine.begin() as conexion:
        await conexion.run_sync(Base.metadata.create_all)

    async with SessionLocal() as session:
        yield session


# 7. CREATE: Crear un libro
@app.post(
    "/api/libros",
    response_model=LibroRespuesta,
    status_code=201
)
async def crear_libro(
    datos: LibroEntrada,
    session: AsyncSession = Depends(obtener_sesion)
):
    nuevo_libro = Libro(
        titulo=datos.titulo,
        anio=datos.anio
    )

    session.add(nuevo_libro)

    await session.commit()
    await session.refresh(nuevo_libro)

    return nuevo_libro


# 8. READ: Consultar todos los libros
@app.get(
    "/api/libros",
    response_model=ListaLibros
)
async def listar_libros(
    session: AsyncSession = Depends(obtener_sesion)
):
    resultado = await session.execute(
        select(Libro).order_by(Libro.id)
    )

    libros = resultado.scalars().all()

    return {"libros": libros}


# 9. READ: Consultar un libro por ID
@app.get(
    "/api/libros/{libro_id}",
    response_model=LibroRespuesta
)
async def consultar_libro(
    libro_id: int,
    session: AsyncSession = Depends(obtener_sesion)
):
    libro = await session.get(Libro, libro_id)

    if libro is None:
        raise HTTPException(
            status_code=404,
            detail="El libro no existe"
        )

    return libro


# 10. UPDATE: Actualizar un libro
@app.put(
    "/api/libros/{libro_id}",
    response_model=LibroRespuesta
)
async def actualizar_libro(
    libro_id: int,
    datos: LibroEntrada,
    session: AsyncSession = Depends(obtener_sesion)
):
    libro = await session.get(Libro, libro_id)

    if libro is None:
        raise HTTPException(
            status_code=404,
            detail="El libro no existe"
        )

    libro.titulo = datos.titulo
    libro.anio = datos.anio

    await session.commit()
    await session.refresh(libro)

    return libro


# 11. DELETE: Eliminar un libro
@app.delete(
    "/api/libros/{libro_id}",
    status_code=204
)
async def eliminar_libro(
    libro_id: int,
    session: AsyncSession = Depends(obtener_sesion)
):
    libro = await session.get(Libro, libro_id)

    if libro is None:
        raise HTTPException(
            status_code=404,
            detail="El libro no existe"
        )

    await session.delete(libro)
    await session.commit()

    return Response(status_code=204)


Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Se está trabajando con Python, FastAPI y SQLAlchemy, utilizando SQLite como base de datos. La implementación debe ser asíncrona y utilizar aiosqlite para evitar que el servidor se bloquee mientras espera las respuestas de la base de datos.|
| **Rol** | Actúa como un desarrollador especializado en Python, FastAPI y bases de datos, encargado de implementar un CRUD completo utilizando sesiones asíncronas|
| **Especificación** |Crear las rutas para registrar, consultar, actualizar y eliminar libros. Se debe utilizar create_async_engine, AsyncSession y await. El sistema debe responder con los códigos HTTP correspondientes: 201 al crear, 200 al consultar y actualizar, 204 al eliminar y 404 cuando un libro no exista. También debe crear automáticamente la tabla al iniciar la aplicación.|
| **Output** | Entregar el código completo del archivo app.py, organizado y funcional, utilizando SQLAlchemy asíncrono y SQLite. El código debe permitir realizar todas las operaciones CRUD y cumplir con los casos de prueba establecidos en la sesión.|

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [8]:
from verificar import probar_http

casos = [{'descripcion': 'Crear un libro',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'Listar los libros',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}]}},
 {'descripcion': 'Actualizar el libro',
  'metodo': 'PUT',
  'ruta': '/api/libros/1',
  'json': {'titulo': 'Rayuela (edición crítica)', 'anio': 1963},
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela (edición crítica)'}},
 {'descripcion': 'Eliminar el libro', 'metodo': 'DELETE', 'ruta': '/api/libros/1', 'estado': 204},
 {'descripcion': 'El libro eliminado ya no existe', 'metodo': 'GET', 'ruta': '/api/libros/1', 'estado': 404}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 5 casos · app.py
✅ 01. Crear un libro
✅ 02. Listar los libros
✅ 03. Actualizar el libro
✅ 04. Eliminar el libro
✅ 05. El libro eliminado ya no existe
------------------------------------------------------------------------------
RESULTADO: 5/5 casos correctos. ¡Resultado logrado!


In [9]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["AsyncSession|async_sessionmaker|aiosqlite"], descripcion="La persistencia es asíncrona"))
resumen(resultados)


✅ La persistencia es asíncrona
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion23.md` a `auditorias/auditoria_sesion23.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue lograr que todas las operaciones del CRUD funcionaran correctamente utilizando sesiones asíncronas. Tuve que revisar cómo utilizar AsyncSession y await para realizar las consultas, guardar los cambios y eliminar los registros sin bloquear el servidor.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí separar el modelo de la base de datos del modelo que recibe los datos de las peticiones. También utilicé una función para obtener la sesión asíncrona y así poder reutilizarla en los diferentes endpoints. Esto me ayudó a tener el código más organizado y fácil de entender.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   Durante la verificación comprendí que no era suficiente con que las operaciones funcionaran, también tenían que devolver exactamente los códigos HTTP solicitados. Por ejemplo, al eliminar un libro debía responder con 204 y después, al intentar consultarlo, debía devolver 404 porque ya no existía.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
